In [44]:
# Untuk gambar dan array
import os
import numpy as np
import pandas as pd
from PIL import Image
from io import BytesIO
import base64

# Untuk ECC encryption (ECIES)
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

# Untuk deep learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, concatenate
from tensorflow.keras.optimizers import Adam

# Untuk hashing
import hashlib

# Konstanta
IMG_SIZE = (64, 64)
NUM_IMAGES = 1000

In [47]:
# Load CSV dan ambil gambar
df = pd.read_csv("results.csv", delimiter='|')
comments = df[' comment'].dropna().head(NUM_IMAGES)

# Fungsi konversi teks ke gambar (dummy grayscale untuk secret image)
def text_to_image(text):
    text_bytes = text.encode('utf-8')
    img_array = np.frombuffer(text_bytes, dtype=np.uint8)
    size = int(np.ceil(np.sqrt(len(img_array))))
    padded = np.pad(img_array, (0, size*size - len(img_array)))
    return padded.reshape((size, size)).astype(np.uint8)

# Preprocessing
cover_images = []
secret_images = []

for text in comments:
    # Dummy image (misalnya semua cover image putih)
    cover = Image.fromarray(np.ones((64, 64, 3), dtype=np.uint8) * 255)
    cover = cover.resize(IMG_SIZE).convert('RGB')
    cover = np.array(cover) / 255.0
    
    # Secret image dari teks
    secret_img = text_to_image(text)
    secret_img = Image.fromarray(secret_img).resize(IMG_SIZE)
    secret_img = np.stack([np.array(secret_img)]*3, axis=-1) / 255.0
    
    cover_images.append(cover)
    secret_images.append(secret_img)

cover_images = np.array(cover_images)
secret_images = np.array(secret_images)


In [48]:
def generate_keys():
    private_key = ec.generate_private_key(ec.SECP256R1(), default_backend())
    public_key = private_key.public_key()
    return private_key, public_key

def derive_shared_key(private_key, peer_public_key):
    shared_key = private_key.exchange(ec.ECDH(), peer_public_key)
    derived_key = HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                       info=b'handshake data', backend=default_backend()).derive(shared_key)
    return derived_key

def encrypt_data(data, key):
    iv = os.urandom(16)
    cipher = Cipher(algorithms.AES(key), modes.CFB(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    ct = encryptor.update(data) + encryptor.finalize()
    return iv + ct

def decrypt_data(encrypted_data, key):
    iv = encrypted_data[:16]
    ct = encrypted_data[16:]
    cipher = Cipher(algorithms.AES(key), modes.CFB(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    return decryptor.update(ct) + decryptor.finalize()


In [51]:
def build_preparation_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    return Model(inputs=inp, outputs=x, name="PreparationNetwork")

def build_hiding_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    for _ in range(4):
        x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(3, (3, 3), padding='same', activation='sigmoid')(x)  # output image 3 channel
    return Model(inputs=inp, outputs=x, name="HidingNetwork")


def build_reveal_network(input_shape):
    inp = Input(shape=input_shape)
    x = Conv2D(65, (3, 3), padding='same', activation='relu')(inp)
    for _ in range(4):
        x = Conv2D(65, (3, 3), padding='same', activation='relu')(x)
    x = Conv2D(3, (3, 3), padding='same', activation='sigmoid')(x)
    return Model(inputs=inp, outputs=x, name="RevealNetwork")


In [53]:
input_shape = (64, 64, 3)

prep_net = build_preparation_network(input_shape)
hide_net = build_hiding_network((64, 64, 65 + 3))
reveal_net = build_reveal_network(input_shape)

# Encode
secret_features = prep_net(Input(shape=input_shape))
combined_input = concatenate([secret_features, Input(shape=input_shape)])
container = hide_net(combined_input)  # output: shape=(None, 64, 64, 3)

# Decode
reconstructed_secret = reveal_net(container)  # input shape cocok

# Stacked Model
encoder_input_secret = Input(shape=input_shape)
encoder_input_cover = Input(shape=input_shape)

prep_out = prep_net(encoder_input_secret)
concat = concatenate([prep_out, encoder_input_cover])
container_out = hide_net(concat)
reveal_out = reveal_net(container_out)

model = Model(inputs=[encoder_input_secret, encoder_input_cover], outputs=reveal_out)
model.compile(optimizer=Adam(), loss='mse')
model.summary()

# Training
model.fit([secret_images, cover_images], secret_images, epochs=500, batch_size=32)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_17      │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PreparationNetwork  │ (None, 64, 64,    │     39,910 │ input_layer_17[0… │
│ (Functional)        │ 65)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_18      │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 64, 64,    │          0 │ PreparationNetwo… │
│ (Concatenate)       │ 68)               │            │ input_layer_18[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ HidingNetwork       │ (None, 64, 64, 3) │    193,963 │ concatenate_4[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RevealNetwork       │ (None, 64, 64, 3) │    155,938 │ HidingNetwork[1]… │
│ (Functional)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 389,811 (1.49 MB)

 Trainable params: 389,811 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - loss: 0.0286
Epoch 2/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 33s 1s/step - loss: 0.0018
Epoch 3/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 996ms/step - loss: 5.5808e-04
Epoch 4/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 989ms/step - loss: 3.0123e-04
Epoch 5/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 1000ms/step - loss: 1.9407e-04
Epoch 6/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 986ms/step - loss: 1.2739e-04
Epoch 7/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 985ms/step - loss: 1.1659e-04
Epoch 8/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 985ms/step - loss: 9.1229e-05
Epoch 9/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 31s 980ms/step - loss: 7.3343e-05
Epoch 10/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 992ms/step - loss: 5.9379e-05
Epoch 11/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 985ms/step - loss: 6.2186e-05
Epoch 12/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 985ms/step - loss: 6.4749e-05
Epoch 13/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 31s 977ms/step - loss: 5.3622e-05
Epoch 14/500
32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 996ms/step - los

In [54]:
def sha512_hash(img_array):
    img_bytes = (img_array * 255).astype(np.uint8).tobytes()
    return hashlib.sha512(img_bytes).hexdigest()

# Contoh hashing
container_img = model.predict([secret_images[:1], cover_images[:1]])[0]
hash_value = sha512_hash(container_img)
print("SHA-512 hash of container image:", hash_value)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step
SHA-512 hash of container image: 77167758058d11001a8f678151cfd47e38fd8ac35c2c8e5958cf22d8f357678b934b66af64272a95f90566256f7599e12152e365b383ea9261fef749b9b91e6e
